# 01 — Data Collection

Сбор всех данных для проекта EquiSense:
- OHLCV (исторические цены)
- Фундаментальные показатели
- Финансовые новости

Источник: **yfinance** (бесплатно, без API ключей)

Данные сохраняются в `data/raw/` в формате Parquet и CSV.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

# Пути
ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW_DIR}")

Raw data directory: /Users/dimabogolaev/EquiSense/data/raw


## 1. Список тикеров

In [2]:
TICKERS = [
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "GOOGL", # Alphabet
    "AMZN",  # Amazon
    "NVDA",  # NVIDIA
    "META",  # Meta
    "TSLA",  # Tesla
    "JPM",   # JPMorgan
    "JNJ",   # Johnson & Johnson
    "V",     # Visa
]

PERIOD = "5y"       # период истории
INTERVAL = "1d"     # дневные свечи

print(f"Тикеров: {len(TICKERS)}")
print(f"Период: {PERIOD}, интервал: {INTERVAL}")

Тикеров: 10
Период: 5y, интервал: 1d


## 2. Загрузка OHLCV данных

In [3]:
def fetch_ohlcv(ticker: str, period: str = "5y", interval: str = "1d") -> pd.DataFrame:
    """Загружает дневные OHLCV данные через yfinance."""
    stock = yf.Ticker(ticker)
    df = stock.history(period=period, interval=interval, auto_adjust=True)
    
    if df.empty:
        print(f"  [!] {ticker}: пустой датафрейм")
        return df
    
    # Нормализуем колонки
    df.index = pd.to_datetime(df.index).tz_localize(None)  # убираем timezone
    df.index.name = "date"
    df.columns = df.columns.str.lower()
    df = df[["open", "high", "low", "close", "volume"]]
    df = df.dropna()
    
    return df


# Загружаем все тикеры
ohlcv_data = {}

for ticker in TICKERS:
    print(f"Загружаю {ticker}...", end=" ")
    df = fetch_ohlcv(ticker, period=PERIOD, interval=INTERVAL)
    
    if not df.empty:
        ohlcv_data[ticker] = df
        # Сохраняем
        out_path = RAW_DIR / f"{ticker}_ohlcv.parquet"
        df.to_parquet(out_path)
        print(f"OK — {len(df)} строк [{df.index[0].date()} → {df.index[-1].date()}]")
    else:
        print("ПРОПУЩЕН")

print(f"\nЗагружено тикеров: {len(ohlcv_data)}/{len(TICKERS)}")

Загружаю AAPL... 

Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
$AAPL: possibly delisted; No price data found  (period=5y)


  [!] AAPL: пустой датафрейм
ПРОПУЩЕН
Загружаю MSFT... 

Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
$MSFT: possibly delisted; No price data found  (period=5y)


  [!] MSFT: пустой датафрейм
ПРОПУЩЕН
Загружаю GOOGL... 

...

  [!] JPM: пустой датафрейм
ПРОПУЩЕН
Загружаю JNJ... 

Failed to get ticker 'JNJ' reason: Expecting value: line 1 column 1 (char 0)
$JNJ: possibly delisted; No price data found  (period=5y)


  [!] JNJ: пустой датафрейм
ПРОПУЩЕН
Загружаю V... 

Failed to get ticker 'V' reason: Expecting value: line 1 column 1 (char 0)
$V: possibly delisted; No price data found  (period=5y)


  [!] V: пустой датафрейм
ПРОПУЩЕН

Загружено тикеров: 0/10


In [4]:
# Быстрый просмотр AAPL
ohlcv_data["AAPL"].tail(10)

KeyError: 'AAPL'

In [ ]:
# Статистика по загруженным данным
summary = []
for ticker, df in ohlcv_data.items():
    summary.append({
        "ticker": ticker,
        "rows": len(df),
        "start": df.index[0].date(),
        "end": df.index[-1].date(),
        "missing_%": round(df.isnull().mean().mean() * 100, 2),
        "last_close": round(df["close"].iloc[-1], 2),
    })

pd.DataFrame(summary).set_index("ticker")

## 3. Фундаментальные данные

In [ ]:
FUNDAMENTAL_FIELDS = {
    "trailingPE":        "pe_ratio",
    "trailingEps":       "eps",
    "returnOnEquity":    "roe",
    "debtToEquity":      "debt_to_equity",
    "revenueGrowth":     "revenue_growth",
    "marketCap":         "market_cap",
    "dividendYield":     "dividend_yield",
    "priceToBook":       "price_to_book",
    "earningsGrowth":    "earnings_growth",
    "currentRatio":      "current_ratio",
    "grossMargins":      "gross_margin",
    "operatingMargins":  "operating_margin",
    "shortName":         "name",
    "sector":            "sector",
    "industry":          "industry",
}


def fetch_fundamentals(ticker: str) -> dict:
    """Получает фундаментальные показатели компании."""
    stock = yf.Ticker(ticker)
    info = stock.info
    
    result = {"ticker": ticker, "fetched_at": datetime.now().isoformat()}
    for yf_field, our_field in FUNDAMENTAL_FIELDS.items():
        result[our_field] = info.get(yf_field)
    
    return result


# Загружаем
fundamentals = []

for ticker in TICKERS:
    print(f"Фундаментал {ticker}...", end=" ")
    try:
        data = fetch_fundamentals(ticker)
        fundamentals.append(data)
        print(f"OK — P/E: {data.get('pe_ratio')}, ROE: {data.get('roe')}")
    except Exception as e:
        print(f"ОШИБКА: {e}")

# Сохраняем
fund_df = pd.DataFrame(fundamentals).set_index("ticker")
fund_df.to_parquet(RAW_DIR / "fundamentals.parquet")
fund_df.to_csv(RAW_DIR / "fundamentals.csv")
print(f"\nСохранено: data/raw/fundamentals.parquet")

In [ ]:
# Просмотр фундаментальных данных
fund_df[["name", "sector", "pe_ratio", "eps", "roe", "debt_to_equity", "revenue_growth", "market_cap"]]

## 4. Финансовые новости

In [ ]:
def fetch_news(ticker: str, max_items: int = 50) -> list[dict]:
    """Получает последние новости по тикеру через yfinance."""
    stock = yf.Ticker(ticker)
    
    news_raw = stock.news or []
    
    news = []
    for item in news_raw[:max_items]:
        content = item.get("content", {})
        news.append({
            "ticker":       ticker,
            "title":        content.get("title", ""),
            "summary":      content.get("summary", ""),
            "source":       content.get("provider", {}).get("displayName", ""),
            "url":          content.get("canonicalUrl", {}).get("url", ""),
            "published_at": content.get("pubDate", ""),
        })
    
    return news


# Загружаем новости
all_news = []

for ticker in TICKERS:
    print(f"Новости {ticker}...", end=" ")
    try:
        news = fetch_news(ticker, max_items=50)
        all_news.extend(news)
        print(f"{len(news)} статей")
    except Exception as e:
        print(f"ОШИБКА: {e}")

# Сохраняем
news_df = pd.DataFrame(all_news)
news_df.to_parquet(RAW_DIR / "news_raw.parquet", index=False)
news_df.to_csv(RAW_DIR / "news_raw.csv", index=False)
print(f"\nВсего статей: {len(news_df)}")
print(f"Сохранено: data/raw/news_raw.parquet")

In [ ]:
# Просмотр новостей
news_df[["ticker", "title", "source", "published_at"]].head(15)

## 5. Проверка сохранённых файлов

In [ ]:
import os

print("Файлы в data/raw/:\n")
files = sorted(RAW_DIR.glob("*"))
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<35} {size_kb:>8.1f} KB")

print(f"\nИтого файлов: {len(files)}")

## 6. Быстрая визуализация

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 6))
axes = axes.flatten()

for i, (ticker, df) in enumerate(ohlcv_data.items()):
    ax = axes[i]
    ax.plot(df.index, df["close"], linewidth=1)
    ax.set_title(ticker, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.grid(alpha=0.3)

plt.suptitle("Closing Prices — 5 Years", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RAW_DIR / "prices_overview.png", dpi=120)
plt.show()
print("График сохранён в data/raw/prices_overview.png")

## Итог

Что было собрано:

| Файл | Описание |
|---|---|
| `{TICKER}_ohlcv.parquet` | Дневные OHLCV данные за 5 лет |
| `fundamentals.parquet` | P/E, EPS, ROE, Debt/Equity, и др. |
| `news_raw.parquet` | Последние новости по каждому тикеру |

**Следующий шаг:** `02_feature_engineering.ipynb` — вычисление технических индикаторов и sentiment.